# Custom RAG: Document Ingestion with Multiple Embedding Models

This notebook:
1. Loads the USO Administration Guide PDF
2. Chunks the document
3. Creates embeddings with **4 different models**
4. Stores in separate ChromaDB collections

## Embedding Models:
- **OpenAI** text-embedding-3-small (1536 dims)
- **Mistral** via HuggingFace
- **Snowflake** snowflake-arctic-embed-l (1024 dims)
- **BAAI** bge-large-en-v1.5 (1024 dims)

In [ ]:
# Setup
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from dotenv import load_dotenv
load_dotenv(Path.cwd().parent / '.env')

import warnings
warnings.filterwarnings('ignore')

print("✓ Setup complete")

In [ ]:
# Imports
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from tqdm.auto import tqdm
import time

from custom_rag_config import (
    EMBEDDING_MODELS,
    CHUNKING_CONFIG,
    VECTOR_STORE_CONFIG,
    SOURCE_DOCS_PATH
)

print("✓ Imports successful")

## Step 1: Load PDF Document

In [ ]:
# Load PDF
pdf_path = Path(SOURCE_DOCS_PATH) / "USO-Administration-Guide.pdf"

if not pdf_path.exists():
    print(f"❌ PDF not found: {pdf_path}")
    print("Please ensure the PDF is in data/source_docs/")
else:
    print(f"📄 Loading PDF: {pdf_path.name}")
    loader = PyPDFLoader(str(pdf_path))
    documents = loader.load()
    
    print(f"✓ Loaded {len(documents)} pages")
    
    # Calculate stats
    total_chars = sum(len(doc.page_content) for doc in documents)
    print(f"✓ Total characters: {total_chars:,}")
    print(f"✓ Average chars per page: {total_chars // len(documents):,}")

## Step 2: Chunk Documents

In [ ]:
# Initialize text splitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNKING_CONFIG["chunk_size"],
    chunk_overlap=CHUNKING_CONFIG["chunk_overlap"],
    separators=CHUNKING_CONFIG["separators"]
)

print(f"📐 Chunking with:")
print(f"   - Chunk size: {CHUNKING_CONFIG['chunk_size']}")
print(f"   - Overlap: {CHUNKING_CONFIG['chunk_overlap']}")
print("\nChunking documents...")

chunks = text_splitter.split_documents(documents)

print(f"\n✓ Created {len(chunks)} chunks")

# Calculate chunk statistics
chunk_sizes = [len(chunk.page_content) for chunk in chunks]
print(f"\nChunk Statistics:")
print(f"   - Min size: {min(chunk_sizes)} chars")
print(f"   - Max size: {max(chunk_sizes)} chars")
print(f"   - Avg size: {sum(chunk_sizes) // len(chunk_sizes)} chars")

# Show example chunk
print(f"\n📝 Example Chunk (first):")
print("=" * 80)
print(chunks[0].page_content[:300] + "...")
print("=" * 80)

## Step 3: Create Embeddings and Vector Stores for Each Model

This will create a separate ChromaDB collection for each embedding model.
This allows fair comparison - same chunks, different embeddings.

In [ ]:
# Helper function to create embeddings
def get_embedding_function(model_config):
    """Create embedding function based on model config"""
    provider = model_config["provider"]
    model_name = model_config["model"]
    
    if provider == "openai":
        return OpenAIEmbeddings(model=model_name)
    elif provider == "huggingface":
        return HuggingFaceEmbeddings(
            model_name=model_name,
            model_kwargs={'device': 'cpu'},  # Use 'cuda' if GPU available
            encode_kwargs={'normalize_embeddings': True}
        )
    else:
        raise ValueError(f"Unknown provider: {provider}")

print("✓ Embedding function helper ready")

In [ ]:
# Create vector stores for all models
vector_stores = {}
ingestion_times = {}

persist_dir = Path(VECTOR_STORE_CONFIG["persist_directory"])
persist_dir.mkdir(parents=True, exist_ok=True)

print(f"📦 Creating vector stores in: {persist_dir}\n")
print("=" * 80)

for model_config in tqdm(EMBEDDING_MODELS, desc="Processing models"):
    model_name = model_config["name"]
    display_name = model_config["display_name"]
    collection_name = model_config["collection_name"]
    
    print(f"\n🔧 Processing: {display_name}")
    print(f"   Collection: {collection_name}")
    print(f"   Dimensions: {model_config['dimensions']}")
    
    try:
        start_time = time.time()
        
        # Get embedding function
        embedding_func = get_embedding_function(model_config)
        print(f"   ✓ Loaded embedding model")
        
        # Create vector store
        print(f"   ⏳ Creating embeddings for {len(chunks)} chunks...")
        vector_store = Chroma.from_documents(
            documents=chunks,
            embedding=embedding_func,
            collection_name=collection_name,
            persist_directory=str(persist_dir)
        )
        
        elapsed = time.time() - start_time
        ingestion_times[model_name] = elapsed
        
        print(f"   ✅ Complete in {elapsed:.1f}s")
        print(f"   💾 Persisted to disk")
        
        # Store reference
        vector_stores[model_name] = vector_store
        
    except Exception as e:
        print(f"   ❌ Failed: {e}")
        continue

print("\n" + "=" * 80)
print(f"\n✅ Ingestion Complete!")
print(f"\nCreated {len(vector_stores)} vector stores")

## Step 4: Verify Vector Stores

In [ ]:
# Test each vector store with a sample query
test_query = "What is USO Administration?"

print(f"🔍 Testing vector stores with query: '{test_query}'\n")
print("=" * 80)

for model_name, vector_store in vector_stores.items():
    model_config = next(m for m in EMBEDDING_MODELS if m["name"] == model_name)
    
    print(f"\n📊 {model_config['display_name']}:")
    
    try:
        # Retrieve similar documents
        results = vector_store.similarity_search(test_query, k=3)
        
        print(f"   ✓ Retrieved {len(results)} documents")
        print(f"\n   Top result preview:")
        print(f"   {results[0].page_content[:150]}...")
        
    except Exception as e:
        print(f"   ❌ Error: {e}")

print("\n" + "=" * 80)

## Step 5: Ingestion Summary

In [ ]:
import pandas as pd

# Create summary table
summary_data = []
for model_config in EMBEDDING_MODELS:
    model_name = model_config["name"]
    if model_name in vector_stores:
        summary_data.append({
            "Model": model_config["display_name"],
            "Provider": model_config["provider"],
            "Dimensions": model_config["dimensions"],
            "Chunks": len(chunks),
            "Collection": model_config["collection_name"],
            "Ingestion Time (s)": f"{ingestion_times.get(model_name, 0):.1f}",
            "Status": "✅ Ready"
        })

summary_df = pd.DataFrame(summary_data)

print("\n📋 INGESTION SUMMARY")
print("=" * 100)
print(summary_df.to_string(index=False))
print("=" * 100)

print(f"\n📦 Vector stores saved to: {persist_dir}")
print(f"\n✅ All vector stores ready for evaluation!")

## Next Steps

✅ **Ingestion complete!** You now have 4 vector stores, one for each embedding model.

**Next:** Run `custom_rag_02_evaluation.ipynb` to:
- Query each vector store
- Apply reranking
- Generate answers
- Calculate Ragas metrics
- Compare embedding models